In [4]:
import pandas as pd

file_path = "Project2_Marketing_Dataset.xlsx"

customers = pd.read_excel(file_path, sheet_name="Customers")
campaigns = pd.read_excel(file_path, sheet_name="Campaigns")
bookings = pd.read_excel(file_path, sheet_name="Bookings")
performance = pd.read_excel(file_path,sheet_name="Campaign_Performance")

customers.shape, campaigns.shape, bookings.shape, performance.shape

((800, 41), (350, 25), (1800, 50), (2000, 18))

In [5]:
customers["customer_id_clean"] = (
    customers["customer_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

bookings["customer_id_clean"] = (
    bookings["customer_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

campaigns["campaign_id_clean"] = (
    campaigns["campaign_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

bookings["campaign_id_clean"] = (
    bookings["campaign_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [6]:
customer_lookup = customers[
    [
        "customer_id_clean",
        "loyalty_tier",
        "pre_defined_segment",
        "preferred_channel",
        "total_lifetime_value",
        "avg_booking_value",
        "total_bookings"
    ]
].rename(
    columns={
        "pre_defined_segment": "customer_profile_segment"
    }
)

campaign_lookup = campaigns[
    [
        "campaign_id_clean",
        "campaign_name",
        "channel",
        "target_segment",
        "objective",
        "budget"
    ]
]

In [7]:
customer_lookup = (
    customer_lookup
    .dropna(subset=["customer_id_clean"])
    .drop_duplicates(subset=["customer_id_clean"])
)

In [8]:
downstream = bookings.merge(
    customer_lookup,
    on="customer_id_clean",
    how="left",
    validate="many_to_one",
    indicator="customer_match"
)

In [9]:
downstream["customer_match"].value_counts(dropna=False)

customer_match
both          1396
left_only      404
right_only       0
Name: count, dtype: int64

In [10]:
campaign_lookup = (
    campaign_lookup
    .dropna(subset=["campaign_id_clean"])
    .drop_duplicates(subset=["campaign_id_clean"])
)

In [11]:
downstream = downstream.merge(
    campaign_lookup,
    on="campaign_id_clean",
    how="left",
    validate="many_to_one",
    indicator="campaign_match"
)

In [12]:
downstream["campaign_match"].value_counts(dropna=False)

campaign_match
both          1792
left_only        8
right_only       0
Name: count, dtype: int64

In [13]:
downstream[
    [
        "booking_id",
        "customer_id",
        "loyalty_tier",
        "customer_profile_segment",
        "campaign_id",
        "campaign_name",
        "channel",
        "objective",
        "status",
        "total_revenue",
        "customer_match",
        "campaign_match"
    ]
].head(10)

,booking_id,customer_id,loyalty_tier,customer_profile_segment,campaign_id,campaign_name,channel,objective,status,total_revenue,customer_match,campaign_match
0,BKG_00229,CUST_0501,Silver,VIP,CMP_0348,Campaign 348,App,Acquisition,No-show,1777.42,both,both
1,BKG_00210,C0427,NaN,VIP,CMP_0334,Campaign 334,Email,Upsell,Completed,1743.30,both,both
2,BKG_00006,cust_0395,Silver,Leisure,CMP_0088,Campaign 88,App,Retention,Cancelled,971.43,both,both
3,BKG_00227,cust_0230,Silver,Long-Stay,CMP_0184,Campaign 184,App,Acquisition,Cancelled,3270.62,both,both
4,BKG_01180,CUST_0554,NaN,Long-Stay,CMP_0044,Campaign 44,Social,Acquisition,No-show,2914.87,both,both
5,BKG_00620,cust_0230,Silver,Long-Stay,CMP_0224,Campaign 224,Affiliate,Acquisition,No-show,419.76,both,both
6,BKG_00596,NaN,NaN,NaN,CMP_0208,Campaign 208,Affiliate,Acquisition,Cancelled,1020.02,left_only,both
7,BKG_00179,C0044,Gold,Long-Stay,CMP_0175,Campaign 175,Email,Retention,Cancelled,4621.82,both,both
8,BKG_00387,cust_0033,Platinum,VIP,CMP_0294,Campaign 294,Social,Acquisition,No-show,926.10,both,both
9,BKG_00619,cust_0678,NaN,Business,CMP_0158,Campaign 158,Affiliate,Acquisition,No-show,8885.56,both,both


In [14]:
downstream.shape

(1800, 65)

In [15]:
downstream.groupby("channel",dropna=False).size().sort_values(ascending=False)

channel
Email        332
Ads          312
Affiliate    294
Social       290
App          285
OTA          279
NaN            8
dtype: int64

In [16]:
downstream.groupby(["channel", "status"], dropna=False).size()

channel    status   
Ads        Cancelled     98
           Completed    107
           No-show      107
Affiliate  Cancelled     99
           Completed     83
           No-show      112
App        Cancelled    100
           Completed     99
           No-show       86
Email      Cancelled    123
           Completed     96
           No-show      113
OTA        Cancelled     82
           Completed    102
           No-show       95
Social     Cancelled    101
           Completed     83
           No-show      106
NaN        Cancelled      2
           Completed      5
           No-show        1
dtype: int64

In [17]:
channel_status = downstream.groupby(["channel", "status"], dropna=False).size().unstack(fill_value=0)

In [18]:
channel_status

status,Cancelled,Completed,No-show
channel,,,
Ads,98,107,107
Affiliate,99,83,112
App,100,99,86
Email,123,96,113
OTA,82,102,95
Social,101,83,106
NaN,2,5,1


In [19]:
channel_status["completion_rate"] = (channel_status["Completed"] / channel_status[["Cancelled", "Completed", "No-show"]].sum(axis=1) * 100).round(2)

In [20]:
channel_status.sort_values("completion_rate", ascending=False)

status,Cancelled,Completed,No-show,completion_rate
channel,,,,
NaN,2,5,1,62.50
OTA,82,102,95,36.56
App,100,99,86,34.74
Ads,98,107,107,34.29
Email,123,96,113,28.92
Social,101,83,106,28.62
Affiliate,99,83,112,28.23


In [21]:
downstream["is_test_record"].value_counts(dropna=False)

is_test_record
N    907
Y    893
Name: count, dtype: int64

In [22]:
analysis_downstream = downstream[downstream["is_test_record"] == "N"].copy()

In [23]:
analysis_downstream.shape

(907, 65)

In [24]:
channel_status_clean = analysis_downstream.groupby(["channel", "status"], dropna=False).size().unstack(fill_value=0)

In [25]:
channel_status_clean

status,Cancelled,Completed,No-show
channel,,,
Ads,54,47,57
Affiliate,49,47,47
App,54,44,39
Email,72,49,59
OTA,39,51,47
Social,48,46,52
NaN,2,4,0


In [26]:
channel_status_clean["completion_rate"] = (channel_status_clean["Completed"] / channel_status_clean[["Cancelled", "Completed", "No-show"]].sum(axis=1) * 100).round(2)

In [27]:
channel_status_clean.sort_values("completion_rate", ascending=False)

status,Cancelled,Completed,No-show,completion_rate
channel,,,,
NaN,2,4,0,66.67
OTA,39,51,47,37.23
Affiliate,49,47,47,32.87
App,54,44,39,32.12
Social,48,46,52,31.51
Ads,54,47,57,29.75
Email,72,49,59,27.22


In [28]:
completed_bookings = analysis_downstream[analysis_downstream["status"] == "Completed"].copy()

In [29]:
completed_bookings.shape

(288, 65)

In [30]:
completed_bookings.groupby(["channel", "currency"], dropna=False)["total_revenue"].sum().round(2)

channel    currency
Ads        CAD          37563.31
           USD         171390.34
Affiliate  CAD          62204.44
           USD          53116.50
App        CAD          55558.15
           USD          90333.50
Email      CAD         113391.05
           USD          66781.02
OTA        CAD          49333.95
           USD         239513.85
Social     CAD         129204.25
           USD          94237.67
NaN        CAD           5354.46
           USD           8312.70
Name: total_revenue, dtype: float64

In [31]:
completed_bookings.groupby("currency")["exchange_rate"].describe()

,count,mean,std,min,25%,50%,75%,max
currency,,,,,,,,
CAD,129.0,0.865610,0.127455,0.7003,0.74240,0.7851,1.0,1.0
USD,159.0,0.867865,0.130308,0.7008,0.73805,0.7987,1.0,1.0


In [32]:
completed_bookings["customer_match"].value_counts(dropna=False)

customer_match
both          224
left_only      64
right_only      0
Name: count, dtype: int64

In [33]:
completed_customer_bookings = completed_bookings[completed_bookings["customer_match"] == "both"].copy()

In [34]:
completed_customer_bookings["loyalty_tier"].value_counts(dropna=False)

loyalty_tier
Platinum    64
Silver      58
NaN         55
Gold        47
Name: count, dtype: int64

In [35]:
customer_bookings = analysis_downstream[analysis_downstream["customer_match"] == "both"].copy()

In [36]:
loyalty_status = customer_bookings.groupby(["loyalty_tier", "status"], dropna=False).size().unstack(fill_value=0)

In [37]:
loyalty_status

status,Cancelled,Completed,No-show
loyalty_tier,,,
Gold,56,47,59
Platinum,54,64,67
Silver,73,58,50
NaN,60,55,64


In [38]:
loyalty_status["completion_rate"] = (loyalty_status["Completed"] / loyalty_status[["Cancelled", "Completed", "No-show"]].sum(axis=1) * 100).round(2)

In [39]:
loyalty_status.sort_values("completion_rate", ascending=False)

status,Cancelled,Completed,No-show,completion_rate
loyalty_tier,,,,
Platinum,54,64,67,34.59
Silver,73,58,50,32.04
NaN,60,55,64,30.73
Gold,56,47,59,29.01


In [40]:
segment_status = customer_bookings.groupby(["customer_profile_segment", "status"], dropna=False).size().unstack(fill_value=0)

In [41]:
segment_status

status,Cancelled,Completed,No-show
customer_profile_segment,,,
Business,42,38,56
Family,57,36,50
Leisure,47,44,39
Long-Stay,53,57,57
VIP,44,49,38


In [42]:
segment_status["completion_rate"] = (segment_status["Completed"] / segment_status[["Cancelled", "Completed", "No-show"]].sum(axis=1) * 100).round(2)

In [43]:
segment_status.sort_values("completion_rate", ascending=False)

status,Cancelled,Completed,No-show,completion_rate
customer_profile_segment,,,,
VIP,44,49,38,37.40
Long-Stay,53,57,57,34.13
Leisure,47,44,39,33.85
Business,42,38,56,27.94
Family,57,36,50,25.17


In [44]:
completed_customer_bookings.groupby(["customer_profile_segment", "currency"], dropna=False)["total_revenue"].sum().round(2)

customer_profile_segment  currency
Business                  CAD          48614.45
                          USD          42960.64
Family                    CAD          43492.64
                          USD          75155.94
Leisure                   CAD          60666.88
                          USD         187858.31
Long-Stay                 CAD         102132.19
                          USD          96303.96
VIP                       CAD         121590.81
                          USD          63627.70
Name: total_revenue, dtype: float64

In [45]:
performance["campaign_id_clean"] = performance["campaign_id"].astype("string").str.strip().str.upper()

In [46]:
performance["is_test_record"].value_counts(dropna=False)

is_test_record
Y    1006
N     994
Name: count, dtype: int64

In [47]:
analysis_performance = performance[performance["is_test_record"] == "N"].copy()

In [48]:
campaigns["is_test_record"].value_counts(dropna=False)

is_test_record
Y    187
N    163
Name: count, dtype: int64

In [49]:
analysis_campaign_lookup = campaigns[campaigns["is_test_record"] == "N"][["campaign_id_clean", "campaign_name", "channel", "target_segment", "objective", "budget"]].copy()

In [50]:
campaign_performance = analysis_performance.merge(analysis_campaign_lookup, on="campaign_id_clean", how="left", validate="many_to_one", indicator="campaign_match")

In [51]:
campaign_performance["campaign_match"].value_counts(dropna=False)

campaign_match
left_only     538
both          456
right_only      0
Name: count, dtype: int64

In [52]:
valid_campaign_performance = campaign_performance[campaign_performance["campaign_match"] == "both"].copy()

In [53]:
channel_performance = valid_campaign_performance.groupby("channel").agg(impressions=("impressions", "sum"), clicks=("clicks", "sum"), conversions=("conversions", "sum"), spend=("spend", "sum")).round(2)

In [54]:
channel_performance

,impressions,clicks,conversions,spend
channel,,,,
Ads,683070,101977,57207,272777.65
Affiliate,711152,96174,45652,275136.30
App,777883,112041,54072,310338.52
Email,880994,124512,64405,364293.21
OTA,613963,95529,48174,267502.44
Social,797511,125873,66567,322388.75


In [55]:
channel_performance["ctr"] = (channel_performance["clicks"] / channel_performance["impressions"] * 100).round(2)

In [56]:
channel_performance["cvr"] = (channel_performance["conversions"] / channel_performance["clicks"] * 100).round(2)

In [57]:
channel_performance["cpc"] = (channel_performance["spend"] / channel_performance["clicks"]).round(2)

In [58]:
channel_performance["cpa"] = (channel_performance["spend"] / channel_performance["conversions"]).round(2)

In [59]:
channel_performance

,impressions,clicks,conversions,spend,ctr,cvr,cpc,cpa
channel,,,,,,,,
Ads,683070,101977,57207,272777.65,14.93,56.10,2.67,4.77
Affiliate,711152,96174,45652,275136.30,13.52,47.47,2.86,6.03
App,777883,112041,54072,310338.52,14.40,48.26,2.77,5.74
Email,880994,124512,64405,364293.21,14.13,51.73,2.93,5.66
OTA,613963,95529,48174,267502.44,15.56,50.43,2.80,5.55
Social,797511,125873,66567,322388.75,15.78,52.88,2.56,4.84


In [60]:
channel_status_clean["total_bookings"] = channel_status_clean[["Cancelled", "Completed", "No-show"]].sum(axis=1)

In [61]:
final_channel_analysis = channel_performance.reset_index().merge(channel_status_clean.reset_index()[["channel", "total_bookings", "Completed", "completion_rate"]], on="channel", how="left", validate="one_to_one")

In [62]:
final_channel_analysis

,channel,impressions,clicks,conversions,spend,ctr,cvr,cpc,cpa,total_bookings,Completed,completion_rate
0,Ads,683070,101977,57207,272777.65,14.93,56.10,2.67,4.77,158,47,29.75
1,Affiliate,711152,96174,45652,275136.30,13.52,47.47,2.86,6.03,143,47,32.87
2,App,777883,112041,54072,310338.52,14.40,48.26,2.77,5.74,137,44,32.12
3,Email,880994,124512,64405,364293.21,14.13,51.73,2.93,5.66,180,49,27.22
4,OTA,613963,95529,48174,267502.44,15.56,50.43,2.80,5.55,137,51,37.23
5,Social,797511,125873,66567,322388.75,15.78,52.88,2.56,4.84,146,46,31.51


In [63]:
valid_downstream = analysis_downstream[analysis_downstream["campaign_id_clean"].isin(analysis_campaign_lookup["campaign_id_clean"])].copy()

In [64]:
valid_downstream.shape

(412, 65)

In [65]:
valid_channel_status = valid_downstream.groupby(["channel", "status"], dropna=False).size().unstack(fill_value=0)

In [66]:
valid_channel_status["total_bookings"] = valid_channel_status[["Cancelled", "Completed", "No-show"]].sum(axis=1)

In [67]:
valid_channel_status["completion_rate"] = (valid_channel_status["Completed"] / valid_channel_status["total_bookings"] * 100).round(2)

In [68]:
valid_channel_status.sort_values("completion_rate", ascending=False)

status,Cancelled,Completed,No-show,total_bookings,completion_rate
channel,,,,,
OTA,15,26,16,57,45.61
Affiliate,23,22,13,58,37.93
Social,22,24,23,69,34.78
Ads,20,25,27,72,34.72
App,25,18,24,67,26.87
Email,41,22,26,89,24.72


In [69]:
final_channel_analysis = channel_performance.reset_index().merge(valid_channel_status.reset_index()[["channel", "total_bookings", "Completed", "completion_rate"]], on="channel", how="left", validate="one_to_one")

In [70]:
final_channel_analysis

,channel,impressions,clicks,conversions,spend,ctr,cvr,cpc,cpa,total_bookings,Completed,completion_rate
0,Ads,683070,101977,57207,272777.65,14.93,56.10,2.67,4.77,72,25,34.72
1,Affiliate,711152,96174,45652,275136.30,13.52,47.47,2.86,6.03,58,22,37.93
2,App,777883,112041,54072,310338.52,14.40,48.26,2.77,5.74,67,18,26.87
3,Email,880994,124512,64405,364293.21,14.13,51.73,2.93,5.66,89,22,24.72
4,OTA,613963,95529,48174,267502.44,15.56,50.43,2.80,5.55,57,26,45.61
5,Social,797511,125873,66567,322388.75,15.78,52.88,2.56,4.84,69,24,34.78


In [71]:
final_channel_analysis["booking_rate_from_conversion"] = (final_channel_analysis["total_bookings"] / final_channel_analysis["conversions"] * 100).round(3)

In [72]:
final_channel_analysis["cost_per_completed_booking"] = (final_channel_analysis["spend"] / final_channel_analysis["Completed"]).round(2)

In [73]:
final_channel_analysis[["channel", "conversions", "total_bookings", "Completed", "booking_rate_from_conversion", "cost_per_completed_booking", "completion_rate"]].sort_values("cost_per_completed_booking")

,channel,conversions,total_bookings,Completed,booking_rate_from_conversion,cost_per_completed_booking,completion_rate
4,OTA,48174,57,26,0.118,10288.56,45.61
0,Ads,57207,72,25,0.126,10911.11,34.72
1,Affiliate,45652,58,22,0.127,12506.20,37.93
5,Social,66567,69,24,0.104,13432.86,34.78
3,Email,64405,89,22,0.138,16558.78,24.72
2,App,54072,67,18,0.124,17241.03,26.87


In [74]:
campaign_performance_summary = valid_campaign_performance.groupby(["campaign_id_clean", "campaign_name", "channel"]).agg(impressions=("impressions", "sum"), clicks=("clicks", "sum"), conversions=("conversions", "sum"), spend=("spend", "sum")).reset_index()

In [75]:
campaign_performance_summary.head()

,campaign_id_clean,campaign_name,channel,impressions,clicks,conversions,spend
0,CMP_0000,Campaign 0,App,9450,1607,710,2919.49
1,CMP_0001,Campaign 1,Affiliate,12349,2877,1318,1003.91
2,CMP_0004,Campaign 4,App,52367,7506,4772,11383.75
3,CMP_0006,Campaign 6,OTA,25326,3612,1848,16165.88
4,CMP_0012,Campaign 12,Ads,34686,4744,2875,13715.04


In [76]:
campaign_booking_status = valid_downstream.groupby(["campaign_id_clean", "status"]).size().unstack(fill_value=0).reset_index()

In [77]:
campaign_booking_status["total_bookings"] = campaign_booking_status[["Cancelled", "Completed", "No-show"]].sum(axis=1)

In [78]:
campaign_booking_status["completion_rate"] = (campaign_booking_status["Completed"] / campaign_booking_status["total_bookings"] * 100).round(2)

In [79]:
campaign_booking_status.head()

status,campaign_id_clean,Cancelled,Completed,No-show,total_bookings,completion_rate
0,CMP_0000,0,2,2,4,50.00
1,CMP_0001,2,1,0,3,33.33
2,CMP_0004,0,0,1,1,0.00
3,CMP_0014,1,0,3,4,0.00
4,CMP_0017,2,1,1,4,25.00


In [80]:
final_campaign_analysis = campaign_performance_summary.merge(campaign_booking_status, on="campaign_id_clean", how="left", validate="one_to_one", indicator="booking_match")

In [81]:
final_campaign_analysis["booking_match"].value_counts(dropna=False)

booking_match
both          136
left_only      16
right_only      0
Name: count, dtype: int64

In [82]:
matched_campaigns = final_campaign_analysis[final_campaign_analysis["booking_match"] == "both"].copy()

In [83]:
campaign_candidates = matched_campaigns[matched_campaigns["total_bookings"] >= 4].copy()

In [84]:
campaign_candidates[["campaign_id_clean", "campaign_name", "channel", "total_bookings", "Completed", "completion_rate", "spend"]].sort_values(["Completed", "completion_rate"], ascending=[False, False]).head(10)

,campaign_id_clean,campaign_name,channel,total_bookings,Completed,completion_rate,spend
16,CMP_0053,Campaign 53,OTA,4.0,4.0,100.00,8603.81
115,CMP_0265,Campaign 265,Affiliate,4.0,4.0,100.00,12124.85
76,CMP_0175,Campaign 175,Email,5.0,4.0,80.00,7344.74
102,CMP_0239,Campaign 239,Affiliate,7.0,4.0,57.14,3450.83
14,CMP_0048,Campaign 48,OTA,5.0,3.0,60.00,7243.57
50,CMP_0131,Campaign 131,Ads,5.0,3.0,60.00,18644.47
88,CMP_0203,Campaign 203,Ads,5.0,3.0,60.00,12792.05
146,CMP_0329,Campaign 329,Ads,5.0,3.0,60.00,7731.10
0,CMP_0000,Campaign 0,App,4.0,2.0,50.00,2919.49
9,CMP_0029,Campaign 29,App,4.0,2.0,50.00,21001.03


In [85]:
campaign_candidates["cost_per_completed_booking"] = (campaign_candidates["spend"] / campaign_candidates["Completed"]).where(campaign_candidates["Completed"] > 0).round(2)

In [86]:
campaign_candidates[["campaign_id_clean", "campaign_name", "channel", "total_bookings", "Completed", "completion_rate", "spend", "cost_per_completed_booking"]].sort_values("cost_per_completed_booking").head(10)

,campaign_id_clean,campaign_name,channel,total_bookings,Completed,completion_rate,spend,cost_per_completed_booking
31,CMP_0081,Campaign 81,Social,5.0,2.0,40.00,61.88,30.94
102,CMP_0239,Campaign 239,Affiliate,7.0,4.0,57.14,3450.83,862.71
0,CMP_0000,Campaign 0,App,4.0,2.0,50.00,2919.49,1459.74
76,CMP_0175,Campaign 175,Email,5.0,4.0,80.00,7344.74,1836.18
89,CMP_0204,Campaign 204,Social,5.0,2.0,40.00,4208.12,2104.06
16,CMP_0053,Campaign 53,OTA,4.0,4.0,100.00,8603.81,2150.95
14,CMP_0048,Campaign 48,OTA,5.0,3.0,60.00,7243.57,2414.52
146,CMP_0329,Campaign 329,Ads,5.0,3.0,60.00,7731.10,2577.03
115,CMP_0265,Campaign 265,Affiliate,4.0,4.0,100.00,12124.85,3031.21
56,CMP_0139,Campaign 139,OTA,4.0,2.0,50.00,6514.24,3257.12


In [87]:
valid_campaign_performance[valid_campaign_performance["campaign_id_clean"] == "CMP_0081"][["date", "impressions", "clicks", "conversions", "spend"]]

,date,impressions,clicks,conversions,spend
265,06/18/2023,12319,635,561,61.88


In [88]:
performance_row_counts = valid_campaign_performance.groupby("campaign_id_clean").size().rename("performance_rows").reset_index()

In [89]:
campaign_candidates = campaign_candidates.merge(performance_row_counts, on="campaign_id_clean", how="left", validate="one_to_one")

In [90]:
reliable_campaigns = campaign_candidates[campaign_candidates["performance_rows"] >= 2].copy()

In [91]:
reliable_campaigns[["campaign_id_clean", "campaign_name", "channel", "performance_rows", "total_bookings", "Completed", "completion_rate", "spend", "cost_per_completed_booking"]].sort_values("cost_per_completed_booking").head(10)

,campaign_id_clean,campaign_name,channel,performance_rows,total_bookings,Completed,completion_rate,spend,cost_per_completed_booking
6,CMP_0053,Campaign 53,OTA,4,4.0,4.0,100.0,8603.81,2150.95
42,CMP_0329,Campaign 329,Ads,2,5.0,3.0,60.0,7731.10,2577.03
33,CMP_0265,Campaign 265,Affiliate,3,4.0,4.0,100.0,12124.85,3031.21
17,CMP_0139,Campaign 139,OTA,4,4.0,2.0,50.0,6514.24,3257.12
27,CMP_0203,Campaign 203,Ads,3,5.0,3.0,60.0,12792.05,4264.02
40,CMP_0310,Campaign 310,Ads,3,4.0,2.0,50.0,9108.17,4554.08
24,CMP_0156,Campaign 156,Social,3,4.0,2.0,50.0,10055.93,5027.96
15,CMP_0131,Campaign 131,Ads,5,5.0,3.0,60.0,18644.47,6214.82
16,CMP_0135,Campaign 135,Ads,5,4.0,2.0,50.0,13741.41,6870.70
39,CMP_0305,Campaign 305,Email,4,5.0,2.0,40.0,13852.27,6926.14


In [92]:
valid_customer_bookings = valid_downstream[valid_downstream["customer_match"] == "both"].copy()

In [93]:
valid_loyalty_status = valid_customer_bookings.groupby(["loyalty_tier", "status"], dropna=False).size().unstack(fill_value=0)

In [94]:
valid_segment_status = valid_customer_bookings.groupby(["customer_profile_segment", "status"], dropna=False).size().unstack(fill_value=0)

In [95]:
valid_loyalty_status["completion_rate"] = (valid_loyalty_status["Completed"] / valid_loyalty_status.sum(axis=1) * 100).round(2)

In [96]:
valid_segment_status["completion_rate"] = (valid_segment_status["Completed"] / valid_segment_status.sum(axis=1) * 100).round(2)

In [97]:
valid_loyalty_status.sort_values("completion_rate", ascending=False)

status,Cancelled,Completed,No-show,completion_rate
loyalty_tier,,,,
Platinum,32,32,28,34.78
NaN,31,26,26,31.33
Gold,24,22,25,30.99
Silver,30,24,27,29.63


In [98]:
valid_segment_status.sort_values("completion_rate", ascending=False)

status,Cancelled,Completed,No-show,completion_rate
customer_profile_segment,,,,
Leisure,21,19,11,37.25
VIP,21,21,15,36.84
Long-Stay,30,32,27,35.96
Business,19,19,21,32.20
Family,26,13,32,18.31


In [99]:
valid_completed_bookings = valid_downstream[valid_downstream["status"] == "Completed"].copy()

In [100]:
valid_channel_revenue = valid_completed_bookings.groupby(["channel", "currency"], dropna=False)["total_revenue"].sum().unstack(fill_value=0).round(2)

In [101]:
valid_segment_revenue = valid_completed_bookings[valid_completed_bookings["customer_match"] == "both"].groupby(["customer_profile_segment", "currency"], dropna=False)["total_revenue"].sum().unstack(fill_value=0).round(2)

In [102]:
valid_channel_revenue

currency,CAD,USD
channel,,
Ads,18559.67,21131.77
Affiliate,11160.39,35547.37
App,17097.88,22301.78
Email,51865.03,20133.05
OTA,26019.34,193453.53
Social,94658.83,50364.54


In [103]:
valid_segment_revenue

currency,CAD,USD
customer_profile_segment,,
Business,15899.87,30940.28
Family,8975.42,10542.32
Leisure,20941.75,11239.66
Long-Stay,50031.89,56324.54
VIP,79200.75,20252.46


In [104]:
output_file = "Project_2_Final_Analysis.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    # 清洗后的明细数据
    valid_downstream.to_excel(
        writer,
        sheet_name="Cleaned Bookings",
        index=False
    )
    
    valid_campaign_performance.to_excel(
        writer,
        sheet_name="Cleaned Performance",
        index=False
    )
    
    # Channel分析
    final_channel_analysis.to_excel(
        writer,
        sheet_name="Channel Summary",
        index=False
    )
    
    valid_channel_revenue.reset_index().to_excel(
        writer,
        sheet_name="Channel Revenue",
        index=False
    )
    
    # Campaign分析
    reliable_campaigns.to_excel(
        writer,
        sheet_name="Campaign Summary",
        index=False
    )
    
    # Customer分析
    valid_loyalty_status.reset_index().to_excel(
        writer,
        sheet_name="Loyalty Summary",
        index=False
    )
    
    valid_segment_status.reset_index().to_excel(
        writer,
        sheet_name="Segment Summary",
        index=False
    )
    
    valid_segment_revenue.reset_index().to_excel(
        writer,
        sheet_name="Segment Revenue",
        index=False
    )

print(f"Excel file exported successfully: {output_file}")

Excel file exported successfully: Project_2_Final_Analysis.xlsx
